In [ ]:
# ============================================================
# CELL 1 - CONFIG + UTILITIES
# Phase 4 CJPE | Model-2 (decoder-only, RoPE+GQA+SwiGLU) + LoRA
# Long-document strategy: SELECTIVE CONTEXT EXTRACTION (head + tail + top-evidence segments) -> gated-attention MIL head
# ============================================================
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")
import re, gc, sys, json, math, time, random, warnings
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings("ignore")
T0 = time.time()

# ------------------------------------------------------------------
# USER SETTINGS  (the only block you normally touch)
# ------------------------------------------------------------------
SMOKE            = False        # True = ~5 min dry run on a tiny subset. RUN THIS ONCE FIRST.
MODE             = "train_predict"   # "train_predict" | "predict_only" (needs best_adapter.pt from an earlier run)
TRAIN_BUDGET_MIN = 660          # total training minutes (LR schedule is time-based, so it always finishes annealing)
MAX_EPOCHS       = 4            # hard cap on passes over the train docs (the sliding-window run peaked after ~3 and then overfit)
SESSION_LIMIT_H  = 11.0         # stop training in time to still run inference before Kaggle's 12h wall
RESERVE_MIN      = 25           # minutes kept free for dev + test inference

# optional manual overrides (otherwise auto-found under /kaggle/input)
TOKENIZER_JSON = None           # e.g. "/kaggle/input/<bpe-dataset>/tokenizer.json"
CKPT_PATH      = None           # e.g. "/kaggle/input/<model2-dataset>/best.pt"

# data
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE4_FILES = {"train": "Phase_4_CJPE/Train/train.json",
                "dev":   "Phase_4_CJPE/Train/dev.json",
                "test":  "Phase_4_CJPE/Test/test.json"}
LOWERCASE = True                    # dev/test are lowercased, train is partly mixed-case -> match the test distribution
EXCLUDE_TEST_IDS_FROM_DEV = True    # all 1,517 test IDs also sit in dev.json; keep them out of model selection / threshold tuning
MAX_CHARS, HEAD_CHARS, TAIL_CHARS = 400_000, 80_000, 240_000   # only affects the ~0.2% monster documents

# selective context extraction (replaces the overlapping-window geometry; details in CELL 2b)
CHUNK        = 1024       # tokens per window fed to Model-2 (its pretraining context length)
STRIDE       = 1024       # only used for SHORT documents that fit the budget and are not cut (consecutive, tail-anchored windows)
N_SEL_WIN    = 2          # selected-evidence windows per document -> budget = N_SEL_WIN * CHUNK tokens (try 3 for a second run)
HEAD_TOK     = 128        # always keep the opening of the judgment (court, parties, issue)
TAIL_TOK     = 384        # always keep the ending of the judgment (reasoning + disposal cues)
SEG_MIN, SEG_MAX = 32, 96 # candidate evidence segments: sentence-aligned, 32..96 tokens
POS_PRIOR    = 1.0        # extra priority for later parts of the judgment (z-score units x relative_position^2)
SEL_NOISE    = 0.5        # train-time jitter of segment priorities (stochastic selection = augmentation); eval/inference is deterministic
SCORER_FOLDS = 4          # the evidence scorer is cross-fitted: a train doc is scored by a model that never saw its label
SCORER_ALPHAS = (3e-6, 1e-5, 3e-5)   # L2 strengths tried for the scorer (best held-out AUC wins)
HASH_BUCKETS = 1_048_573  # hashed-bigram feature space of the scorer (prime)
NB = 8                    # position buckets of the MIL head (first slot -> "first", last slot -> "anchor window")

# adapter
ADAPTER = "lora"          # "lora" (additive) | "lorma_plus" (multiplicative, (I + s*BA) W0 - see notes)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.10
LORA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "w_gate", "w_up", "w_down")
BACKBONE_DROPOUT = 0.1    # Model-2 was pretrained with 0.2; lighter for fine-tuning

# optimisation
MICRO_DOCS, ACCUM = 4, 4              # 4 docs x <=2 windows = <=8 windows / micro-batch; 16 docs / optimizer step
LR_LORA, LR_HEAD, WD_HEAD = 3e-4, 1e-3, 0.01
WARMUP_STEPS, MIN_LR_RATIO, GRAD_CLIP = 50, 0.05, 1.0
AUX_W = 0.3                           # weight of the per-window auxiliary loss (deep supervision for the MIL head)
LOG_EVERY, EVAL_EVERY_MIN, CKPT_EVERY_MIN = 20, 10.0, 15.0   # epochs are ~3x shorter now -> validate more often
N_QUICK = 1000                        # dev docs used for the periodic quick validation (400 was too noisy for picking a checkpoint)
INFER_BS, DOC_BS = 32, 64
SEED = 1234

if SMOKE:
    TRAIN_BUDGET_MIN, MAX_EPOCHS, RESERVE_MIN = 3, 1, 0
    EVAL_EVERY_MIN, CKPT_EVERY_MIN, N_QUICK = 1.0, 1.0, 40
    N_SMOKE = {"train": 160, "dev": 90, "test": 40}

# ------------------------------------------------------------------
# paths / device
# ------------------------------------------------------------------
IS_KAGGLE  = os.path.isdir("/kaggle/working")
INPUT_ROOT = "/kaggle/input"
WORK  = Path("/kaggle/working" if IS_KAGGLE else ".") / ("phase4_model2_lora_selective" + ("_smoke" if SMOKE else ""))
CACHE, CKPT, OUT, SUB = WORK / "cache", WORK / "ckpt", WORK / "out", WORK / "submission"
for _d in (CACHE, CKPT, OUT, SUB):
    _d.mkdir(parents=True, exist_ok=True)

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP  = DEVICE.type == "cuda"
AMP_DTYPE = torch.float16        # T4 / P100 have no bf16
OOM_ERR  = getattr(torch.cuda, "OutOfMemoryError", RuntimeError)

def amp_ctx():
    return torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP)

def log(*a):
    print(*a, flush=True)

def fmt_t(sec):
    sec = int(max(sec, 0)); return f"{sec // 3600}h{(sec % 3600) // 60:02d}m"

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(SEED)

def read_json(path):
    try:
        with open(path, encoding="utf-8") as f: return json.load(f)
    except Exception:
        return None

def find_files(name, roots):
    hits = []
    for r in roots:
        if r and os.path.isdir(r):
            for d, _, fs in os.walk(r):
                if name in fs: hits.append(os.path.join(d, name))
    return hits

def atomic_save(obj, path):
    tmp = str(path) + ".tmp"; torch.save(obj, tmp); os.replace(tmp, path)

log(f"device={DEVICE} | amp={USE_AMP} | SMOKE={SMOKE} | MODE={MODE} | work dir={WORK}")
if DEVICE.type == "cuda":
    log("GPU:", torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB")


In [ ]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF")

print("Token loaded:", bool(os.environ.get("HF_TOKEN")))
print("Token prefix:", os.environ["HF_TOKEN"][:3])

In [ ]:
# ============================================================
# CELL 2 - LOCATE MODEL/TOKENIZER, LOAD CJPE DATA, CLEAN, TOKENISE (cached), AUDIT
# ============================================================
from tokenizers import Tokenizer

# ---------- locate tokenizer + Model-2 checkpoint (attached Kaggle inputs) ----------
def locate_tokenizer():
    if TOKENIZER_JSON: return TOKENIZER_JSON
    hits = find_files("tokenizer.json", [INPUT_ROOT, str(WORK)])
    ranked = []
    for h in hits:
        meta = read_json(os.path.join(os.path.dirname(h), "meta.json")) or {}
        ranked.append((0 if meta.get("tokenizer_type") == "true_utf8_byte_level_bpe" else 1, h))
    if not ranked: raise FileNotFoundError("tokenizer.json not found - attach the BPE output dataset")
    return sorted(ranked)[0][1]

def locate_checkpoint():
    if CKPT_PATH: return CKPT_PATH
    best = None
    for h in find_files("best.pt", [INPUT_ROOT]):
        meta = read_json(os.path.join(os.path.dirname(h), "best_meta.json")) or {}
        vl = meta.get("val_loss", 1e9)
        if "clean_bpe32k" in str(meta.get("tag", "clean_bpe32k")) and (best is None or vl < best[0]):
            best = (vl, h)
    if best: log(f"best.pt val_loss={best[0]:.4f}"); return best[1]
    hits = find_files("latest.pt", [INPUT_ROOT])
    if hits: log("best.pt not found - falling back to latest.pt"); return hits[0]
    raise FileNotFoundError("Model-2 best.pt / latest.pt not found - attach the Model-2 checkpoint dataset")

TOK_PATH  = locate_tokenizer()
MODEL_CKPT = locate_checkpoint()
tokenizer = Tokenizer.from_file(TOK_PATH)
tokenizer.no_truncation(); tokenizer.no_padding()
VOCAB = tokenizer.get_vocab_size()
assert VOCAB < 65536, "token ids are cached as uint16"
log(f"tokenizer : {TOK_PATH} (vocab {VOCAB})")
log(f"checkpoint: {MODEL_CKPT}")

# ---------- text preparation (mirrors the cleaning used for the pretraining corpus) ----------
_ZW   = re.compile(r"[\u200b\u200c\u200d\u2060\ufeff]")
_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
_HYPH = re.compile(r"(?<=\w)-\n(?=\w)")
_WRAP = re.compile(r"(?<=\w)\n(?=\w)")
_SP   = re.compile(r"[ \t]+")
_NLSP = re.compile(r"[ \t]*\n[ \t]*")
_NL3  = re.compile(r"\n{3,}")

def prep_text(t):
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = _CTRL.sub(" ", _ZW.sub("", t)).replace("\ufffd", " ")
    t = _HYPH.sub("", t)
    t = _WRAP.sub(" ", t)                 # undo hard line-wraps (dev/test are wrapped at ~50 chars)
    t = _NL3.sub("\n\n", _NLSP.sub("\n", _SP.sub(" ", t))).strip()
    if len(t) > MAX_CHARS:                # keep the beginning and (mostly) the end of giant judgments
        t = t[:HEAD_CHARS] + "\n" + t[-TAIL_CHARS:]
    return t.lower() if LOWERCASE else t

class TokDS:
    """tokenised split stored as one flat uint16 array + offsets"""
    def __init__(self, ids, labels, flat, offs, has_upper):
        self.ids, self.labels, self.flat, self.offs, self.has_upper = list(ids), np.asarray(labels), flat, offs, has_upper
        self.n_tok = np.diff(offs)
    def __len__(self): return len(self.ids)
    def slice(self, i, s, l): return self.flat[self.offs[i] + s: self.offs[i] + s + l]
    def subset(self, idx):
        idx = np.asarray(idx)
        parts = [self.flat[self.offs[i]:self.offs[i + 1]] for i in idx]
        offs = np.concatenate([[0], np.cumsum([len(p) for p in parts])]).astype(np.int64)
        return TokDS([self.ids[i] for i in idx], self.labels[idx], np.concatenate(parts), offs, self.has_upper[idx])

SIG = f"{'smoke' if SMOKE else 'full'}_lc{int(LOWERCASE)}_ex{int(EXCLUDE_TEST_IDS_FROM_DEV)}_v{VOCAB}"

def _hf_token():
    t = os.environ.get("HF_TOKEN")
    if t: return t
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF")
    except Exception:
        return None

def _download(split):
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=PHASE4_FILES[split], repo_type="dataset", token=_hf_token())

def _records(split):
    obj = read_json(_download(split))
    if isinstance(obj, dict):
        for k in ("data", "records", "examples"):
            if isinstance(obj.get(k), list): obj = obj[k]; break
    return obj

def _tokenise(texts, bs=128):
    flat, lens = [], []
    for i in range(0, len(texts), bs):
        for e in tokenizer.encode_batch(texts[i:i + bs], add_special_tokens=False):
            a = np.asarray(e.ids, dtype=np.uint16); flat.append(a); lens.append(len(a))
        if (i // bs) % 40 == 0: log(f"    tokenised {min(i + bs, len(texts)):,}/{len(texts):,}")
    return np.concatenate(flat), np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)

def build_split(split, records, with_labels):
    t0 = time.time()
    ids = [str(r["id"]) for r in records]
    raw = [r.get("text") or "" for r in records]
    has_upper = np.array([t[:5000] != t[:5000].lower() for t in raw])
    labels = np.array([int(r["label"]) for r in records]) if with_labels else np.full(len(records), -1)  # test labels are NEVER read
    flat, offs = _tokenise([prep_text(t) for t in raw])
    log(f"  {split}: {len(ids):,} docs, {len(flat)/1e6:.1f}M tokens in {(time.time()-t0)/60:.1f} min")
    return TokDS(ids, labels, flat, offs, has_upper)

def save_split(split, ds):
    np.savez(CACHE / f"{split}_{SIG}.npz", ids=np.array(ds.ids), labels=ds.labels, flat=ds.flat, offs=ds.offs, has_upper=ds.has_upper)

def load_split(split):
    p = CACHE / f"{split}_{SIG}.npz"
    if not p.exists():                                   # the tokenised cache of an earlier run (attached as input) is reused: saves ~7 min
        hits = find_files(p.name, [INPUT_ROOT])
        if not hits: return None
        p = Path(hits[0])
    z = np.load(p)
    return TokDS(z["ids"].tolist(), z["labels"], z["flat"], z["offs"], z["has_upper"])

def get_all_splits():
    cached = {s: load_split(s) for s in ("train", "dev", "test")}
    if all(v is not None for v in cached.values()):
        log("tokenised splits loaded from cache"); return cached
    rng = np.random.default_rng(SEED)
    log("downloading / reading CJPE json ...")
    test_rec = _records("test")
    test_ids = {str(r["id"]) for r in test_rec}              # IDs only - test labels are not touched
    test_rec = [{"id": r["id"], "text": r.get("text")} for r in test_rec]
    dev_rec = _records("dev")
    n_overlap = sum(str(r["id"]) in test_ids for r in dev_rec)
    log(f"!! dev/test ID overlap: {n_overlap:,} of {len(test_ids):,} test IDs are also in dev.json")
    if EXCLUDE_TEST_IDS_FROM_DEV:
        dev_rec = [r for r in dev_rec if str(r["id"]) not in test_ids]
        log(f"   -> dev reduced to {len(dev_rec):,} docs that are NOT test cases (used for selection + threshold tuning)")
    tr_rec = _records("train")
    if SMOKE:
        pick_ = lambda recs, n: [recs[i] for i in rng.choice(len(recs), min(n, len(recs)), replace=False)]
        tr_rec, dev_rec, test_rec = pick_(tr_rec, N_SMOKE["train"]), pick_(dev_rec, N_SMOKE["dev"]), pick_(test_rec, N_SMOKE["test"])
    out = {"train": build_split("train", tr_rec, True), "dev": build_split("dev", dev_rec, True), "test": build_split("test", test_rec, False)}
    del tr_rec, dev_rec, test_rec; gc.collect()
    for s, d in out.items(): save_split(s, d)
    return out

SPLITS = get_all_splits()
ds_train, ds_dev, ds_test = SPLITS["train"], SPLITS["dev"], SPLITS["test"]
assert len(set(ds_test.ids)) == len(ds_test.ids), "duplicate test ids"

# ---------- window geometry (pure numpy) ----------
def all_starts(n, W=CHUNK, stride=STRIDE):
    """start offsets of overlapping windows; the LAST window is anchored to the end of the document,
    so the tail of the judgment (reasoning + disposal) is always covered by a full-length window"""
    if n <= W: return np.array([0], dtype=np.int64)
    s = np.arange(0, n - W + 1, stride, dtype=np.int64)
    if s[-1] != n - W: s = np.append(s, n - W)
    return s

def pick(n_win, K, rng=None):
    """coverage-preserving selection of K of n_win windows: first 1 + last 2 always kept,
    the other K-3 are one-per-segment over the middle (random inside a segment when training, centred at eval)"""
    if n_win <= K: return np.arange(n_win)
    if K == 1: return np.array([n_win - 1])
    if K == 2: return np.array([0, n_win - 1])
    keep = {0, n_win - 2, n_win - 1}
    m = K - 3
    if m > 0:
        lo, hi = 1, n_win - 3
        edges = np.linspace(lo, hi + 1, m + 1)
        for i in range(m):
            a = int(np.ceil(edges[i])); b = max(int(np.ceil(edges[i + 1])), a + 1)
            keep.add(min(int(rng.integers(a, b)) if rng is not None else (a + b - 1) // 2, hi))
    return np.array(sorted(keep))

def bucket_ids(g, n_all):
    """position embedding id of a window: 0 = first, NB+1 = last, 1..NB = relative position in between"""
    g = np.asarray(g)
    if n_all <= 1: return np.zeros(len(g), dtype=np.int64)
    b = 1 + np.minimum((g / (n_all - 1) * NB).astype(np.int64), NB - 1)
    b = np.where(g == n_all - 1, NB + 1, b)
    return np.where(g == 0, 0, b).astype(np.int64)

# ---------- audit ----------
def audit(name, ds):
    nt = ds.n_tok
    log(f"{name:9s} docs={len(ds):6,} | tokens mean={nt.mean():7.0f} median={np.median(nt):6.0f} p90={np.percentile(nt,90):7.0f} "
        f"p99={np.percentile(nt,99):8.0f} | {(nt > N_SEL_WIN * CHUNK).mean()*100:3.0f}% longer than the {N_SEL_WIN * CHUNK}-token budget (selection applies) | "
        f"{ds.has_upper.mean()*100:3.0f}% mixed-case in source")
    if (ds.labels >= 0).all():
        c = np.bincount(ds.labels, minlength=2)
        log(f"{'':9s} labels: class0={c[0]:,} ({100*c[0]/c.sum():.1f}%)  class1={c[1]:,} ({100*c[1]/c.sum():.1f}%)  imbalance={c.max()/max(c.min(),1):.2f}")
log("=" * 100)
audit("train", ds_train); audit("dev(clean)", ds_dev); audit("test", ds_test)
log("=" * 100)

cnt = np.bincount(ds_train.labels, minlength=2).astype(np.float64)
CLASS_W = torch.tensor(cnt.sum() / (2.0 * cnt), dtype=torch.float32, device=DEVICE)   # "balanced" inverse-frequency weights
log(f"class weights (train, inverse-frequency): {CLASS_W.tolist()}")


In [ ]:
# ============================================================
# CELL 2b - SELECTIVE CONTEXT EXTRACTION  (replaces the sliding-window geometry)
#
#   judgment (up to 50k tokens)
#     -> sentence-aligned candidate segments (32..96 tokens)
#     -> evidence score per segment from a fast n-gram linear scorer (cross-fitted, so TRAIN selection
#        looks exactly like DEV/TEST selection: no segment is ever scored by a model that saw its document's label)
#     -> keep  [opening 128 tok] + [ending 384 tok]  +  the highest-priority segments up to N_SEL_WIN * 1024 tokens
#        (priority = per-document z-score of the evidence score + late-judgment prior; jittered at train time)
#     -> N_SEL_WIN windows of <= 1024 tokens in ORIGINAL order, ordered [evidence windows ..., anchor window];
#        the anchor window ends with the true end of the judgment.
#   Documents that already fit into the budget are NOT cut: they become consecutive windows (tail-anchored).
#   The LoRA backbone + gated-attention MIL head downstream are unchanged.
# ============================================================
from scipy import sparse as sp
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import normalize
from sklearn.metrics import roc_auc_score as _auc

assert HEAD_TOK + TAIL_TOK + 3 * SEG_MAX < CHUNK, "HEAD_TOK + TAIL_TOK leave no room for evidence in the anchor window"
HB = HASH_BUCKETS                       # hashed-bigram buckets (prime)
NF = VOCAB + HB                         # feature space: BPE unigrams + hashed BPE bigrams
MIN_DF = 2 if SMOKE else 3
SC_EPOCHS = 8
SIG2 = f"{SIG}_seg{SEG_MIN}-{SEG_MAX}_f{SCORER_FOLDS}_hb{HB}_s{SEED}"

# ---------------------------------------------------------------- 1. sentence-final tokens (works on the BPE vocab, no text needed)
def _boundary_table():
    tab = np.zeros(VOCAB, dtype=bool)
    for i in range(VOCAB):
        s = tokenizer.decode([i], skip_special_tokens=False)
        tab[i] = ("\n" in s) or s.rstrip(" ").endswith((".", "?", "!"))
    return tab

BOUND = _boundary_table()
SEP = np.asarray(tokenizer.encode("\n", add_special_tokens=False).ids[:1], dtype=np.uint16)   # paragraph break between non-adjacent excerpts
log(f"selective context: {int(BOUND.sum()):,} sentence-final tokens in the vocab | separator token id {SEP.tolist()} | "
    f"budget {N_SEL_WIN} x {CHUNK} tokens (head {HEAD_TOK} + tail {TAIL_TOK} always kept)")

def segment_ends(ids):
    """exclusive end offsets of consecutive segments of SEG_MIN..SEG_MAX tokens, cut after a sentence-final token when possible"""
    n = len(ids)
    if n <= SEG_MIN: return np.array([n], dtype=np.int64)
    cand = np.flatnonzero(BOUND[ids]) + 1
    ends, s = [], 0
    while s < n:
        j = int(np.searchsorted(cand, s + SEG_MIN))
        e = int(cand[j]) if j < len(cand) else n
        e = min(e, s + SEG_MAX, n)
        ends.append(e); s = e
    if len(ends) > 1 and ends[-1] - ends[-2] < SEG_MIN // 2:      # merge a tiny trailing fragment into its neighbour
        ends.pop(-2)
    return np.asarray(ends, dtype=np.int64)

class SegIndex:
    """ptr[d]:ptr[d+1] = the segments of document d; start = token offset inside the doc; score = signed evidence score"""
    def __init__(self, ptr, start, length, score):
        self.ptr, self.start, self.length, self.score = ptr, start, length, score

def build_seg_index(ds):
    ptr, starts, lens = [0], [], []
    for d in range(len(ds)):
        e = segment_ends(ds.flat[ds.offs[d]:ds.offs[d + 1]])
        s = np.concatenate([[0], e[:-1]])
        starts.append(s); lens.append(e - s); ptr.append(ptr[-1] + len(e))
    start = np.concatenate(starts).astype(np.int64); length = np.concatenate(lens).astype(np.int32)
    return SegIndex(np.asarray(ptr, dtype=np.int64), start, length, np.zeros(len(start), dtype=np.float32))

# ---------------------------------------------------------------- 2. binary hashed n-gram features + cross-fitted linear evidence scorer
def _bigram_cols(a):
    return VOCAB + (a[:-1] * 1_000_003 + a[1:]) % HB

def matrix_from_arrays(arrs):
    """binary (presence) unigram+bigram features, one CSR row per token array"""
    rows, ptr = [], [0]
    for ids in arrs:
        a = np.asarray(ids).astype(np.int64)
        f = np.unique(np.concatenate([a, _bigram_cols(a)]) if len(a) > 1 else a)
        rows.append(f.astype(np.int32)); ptr.append(ptr[-1] + len(f))
    return sp.csr_matrix((np.ones(ptr[-1], dtype=np.float32), np.concatenate(rows), np.asarray(ptr, dtype=np.int64)), shape=(len(rows), NF))

def _doc_arrays(ds, idx):
    for d in idx: yield ds.flat[ds.offs[d]:ds.offs[d + 1]]

def _sgd(alpha, seed):
    return SGDClassifier(loss="log_loss", penalty="l2", alpha=alpha, max_iter=SC_EPOCHS, tol=None, average=True,
                         class_weight="balanced", random_state=seed)

def fit_scorer():
    t0 = time.time()
    X = matrix_from_arrays(_doc_arrays(ds_train, range(len(ds_train))))
    N = X.shape[0]
    dfreq = np.bincount(X.indices, minlength=NF)
    idf = (np.log((1 + N) / (1 + dfreq)) + 1.0).astype(np.float32); idf[dfreq < MIN_DF] = 0.0
    Xt = normalize(X @ sp.diags(idf), copy=False); del X
    y = ds_train.labels
    folds = list(StratifiedKFold(SCORER_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(N), y))
    tr0, te0 = folds[0]
    res = []
    for a in SCORER_ALPHAS:
        clf = _sgd(a, SEED).fit(Xt[tr0], y[tr0]); res.append((_auc(y[te0], clf.decision_function(Xt[te0])), a))
        log(f"  scorer alpha={a:.0e}: held-out AUC {res[-1][0]:.4f}")
    alpha = max(res)[1]
    W, fold_of = [], np.zeros(N, dtype=np.int64)
    for k, (tr, te) in enumerate(folds):
        clf = _sgd(alpha, SEED + k).fit(Xt[tr], y[tr]); W.append(clf.coef_[0].astype(np.float32)); fold_of[te] = k
        log(f"  scorer fold {k}: out-of-fold AUC {_auc(y[te], clf.decision_function(Xt[te])):.4f}")
    del Xt; gc.collect()
    log(f"evidence scorer fitted (alpha={alpha:.0e}, {SCORER_FOLDS} folds) in {(time.time() - t0) / 60:.1f} min")
    return dict(W=np.stack(W), idf=idf, fold_of=fold_of, alpha=alpha)

def assign_scores(ds, seg, doc_ids, q):
    """segment score = sum over its tokens of  q[unigram] + q[bigram],  q = idf * weight  (signed: >0 -> class 1, <0 -> class 0)"""
    for d in doc_ids:
        a = ds.flat[ds.offs[d]:ds.offs[d + 1]].astype(np.int64)
        tw = q[a]
        if len(a) > 1: tw[:-1] += q[_bigram_cols(a)]
        lo, hi = seg.ptr[d], seg.ptr[d + 1]
        seg.score[lo:hi] = np.add.reduceat(tw, seg.start[lo:hi])

def _cache_find(name):
    p = CACHE / name
    if p.exists(): return p
    hits = find_files(name, [INPUT_ROOT])
    return Path(hits[0]) if hits else None

def build_or_load():
    names = {s: f"seg_{s}_{SIG2}.npz" for s in ("train", "dev", "test")}
    paths = {s: _cache_find(n) for s, n in names.items()}; psc = _cache_find(f"scorer_{SIG2}.npz")
    if all(paths.values()) and psc is not None:
        segs = {}
        for s, p in paths.items():
            z = np.load(p); segs[s] = SegIndex(z["ptr"], z["start"], z["length"], z["score"])
        z = np.load(psc); sc = dict(W=z["W"], idf=z["idf"], fold_of=z["fold_of"], alpha=float(z["alpha"]))
        log("segment index + evidence scores loaded from cache"); return segs, sc
    t0 = time.time()
    segs = {"train": build_seg_index(ds_train), "dev": build_seg_index(ds_dev), "test": build_seg_index(ds_test)}
    log(f"segmented: " + " | ".join(f"{s} {len(v.start):,} segments (mean {v.length.mean():.0f} tok)" for s, v in segs.items()) + f" | {(time.time() - t0) / 60:.1f} min")
    sc = fit_scorer()
    q_all = sc["W"] * sc["idf"][None]
    for k in range(SCORER_FOLDS):                                                    # train: each doc scored by the model that did NOT see it
        assign_scores(ds_train, segs["train"], np.flatnonzero(sc["fold_of"] == k), q_all[k])
    q_avg = q_all.mean(0)                                                            # dev/test: average of the fold models
    assign_scores(ds_dev, segs["dev"], range(len(ds_dev)), q_avg); assign_scores(ds_test, segs["test"], range(len(ds_test)), q_avg)
    for s, v in segs.items():
        np.savez(CACHE / names[s], ptr=v.ptr, start=v.start, length=v.length, score=v.score)
    np.savez(CACHE / f"scorer_{SIG2}.npz", **sc)
    log(f"selective-context precompute done in {(time.time() - t0) / 60:.1f} min")
    return segs, sc

SEG, SCORER = build_or_load()
ds_train.seg, ds_dev.seg, ds_test.seg = SEG["train"], SEG["dev"], SEG["test"]

# ---------------------------------------------------------------- 3. the selector
def _assemble(ds, d, idxs):
    seg = ds.seg; base = int(ds.offs[d]); lo = seg.ptr[d]
    parts, prev_end = [], -1
    for j in np.sort(np.asarray(idxs)):
        s = int(seg.start[lo + j]); l = int(seg.length[lo + j])
        if parts and s != prev_end: parts.append(SEP)                              # jump in the judgment -> paragraph break
        parts.append(ds.flat[base + s: base + s + l]); prev_end = s + l
    return np.concatenate(parts)

def select_context(ds, d, rng=None, mode="learned"):
    """token arrays (<= CHUNK each), ordered [evidence windows ..., anchor window]; the anchor ends with the end of the judgment.
       rng=None -> deterministic (eval / inference); rng given -> priorities jittered by SEL_NOISE (train-time augmentation).
       mode='random' = same budget and same always-kept head/tail, but random segments (diagnostic baseline)."""
    n = int(ds.n_tok[d]); base = int(ds.offs[d]); cap = N_SEL_WIN * CHUNK
    if n <= cap:                                                                   # fits: nothing to drop
        return [ds.flat[base + int(s): base + int(s) + min(CHUNK, n - int(s))] for s in all_starts(n, CHUNK, CHUNK)]
    seg = ds.seg; lo, hi = seg.ptr[d], seg.ptr[d + 1]
    ln = seg.length[lo:hi].astype(np.int64); m = len(ln)
    pos = np.arange(m) / max(m - 1, 1)
    if mode == "random":
        pri = (rng if rng is not None else np.random.default_rng(d)).standard_normal(m)
    else:
        mag = np.abs(seg.score[lo:hi]) / np.sqrt(ln)                              # evidence density (sign-agnostic)
        pri = (mag - mag.mean()) / (mag.std() + 1e-6) + POS_PRIOR * pos ** 2
        if rng is not None and SEL_NOISE > 0: pri = pri + SEL_NOISE * rng.standard_normal(m)
    must = np.zeros(m, dtype=bool)
    must[: int(np.searchsorted(np.cumsum(ln), HEAD_TOK)) + 1] = True
    must[m - (int(np.searchsorted(np.cumsum(ln[::-1]), TAIL_TOK)) + 1):] = True
    groups = [list(np.flatnonzero(must))] + [[] for _ in range(N_SEL_WIN - 1)]     # group 0 = anchor window
    room = [CHUNK - int(ln[must].sum()) - int(must.sum())] + [CHUNK] * (N_SEL_WIN - 1)   # 1 separator token reserved per segment
    taken = must.copy()
    for j in np.argsort(-pri):
        if taken[j]: continue
        need = int(ln[j]) + 1
        for g in range(N_SEL_WIN):                                                 # best segments go to the anchor window first
            if room[g] >= need:
                groups[g].append(int(j)); room[g] -= need; taken[j] = True; break
        else:
            if max(room) < SEG_MIN // 2: break                                     # every window is full
    wins = [_assemble(ds, d, g) for g in groups[1:][::-1] if g] + [_assemble(ds, d, groups[0])]
    return wins

# ---------------------------------------------------------------- 4. diagnostics: how much signal survives the selection?
def selection_report(ds, name="dev"):
    """AUC of the (full-document) linear scorer when it only sees the selected context vs. naive alternatives with the SAME token budget."""
    w_idf = SCORER["W"].mean(0)
    def auc_of(arrs):
        Xt = normalize(matrix_from_arrays(arrs) @ sp.diags(SCORER["idf"]), copy=False)
        return _auc(ds.labels, np.asarray(Xt @ w_idf).ravel())
    cap = N_SEL_WIN * CHUNK; idx = range(len(ds)); rng = np.random.default_rng(SEED)
    rows = [("full document (upper bound for this scorer)", auc_of(_doc_arrays(ds, idx))),
            (f"selected context ({N_SEL_WIN} x {CHUNK}, this notebook)", auc_of(np.concatenate(select_context(ds, d)) for d in idx)),
            (f"random segments, same budget + head/tail", auc_of(np.concatenate(select_context(ds, d, rng, mode='random')) for d in idx)),
            (f"last {cap} tokens only (naive truncation)", auc_of(ds.flat[max(ds.offs[d], ds.offs[d + 1] - cap):ds.offs[d + 1]] for d in idx)),
            (f"first {cap} tokens only (naive truncation)", auc_of(ds.flat[ds.offs[d]:min(ds.offs[d + 1], ds.offs[d] + cap)] for d in idx))]
    kept = np.array([sum(len(w) for w in select_context(ds, d)) for d in idx]); tot = ds.n_tok
    log(f"\nSELECTIVE CONTEXT CHECK on {name} ({len(ds):,} docs): tokens kept {kept.sum() / tot.sum() * 100:.1f}% overall | "
        f"{(tot > cap).mean() * 100:.0f}% of docs exceeded the {cap}-token budget (the rest are kept whole)")
    for r, a in rows: log(f"   linear-scorer AUC | {r:52s} | {a:.4f}")
    return rows

def show_example(ds, d=0, top=4):
    wins = select_context(ds, d); seg = ds.seg; lo, hi = seg.ptr[d], seg.ptr[d + 1]
    order = np.argsort(-np.abs(seg.score[lo:hi]) / np.sqrt(seg.length[lo:hi]))[:top]
    log(f"\nexample doc {ds.ids[d]}: {ds.n_tok[d]:,} tokens -> {len(wins)} windows of {[len(w) for w in wins]} tokens | label {ds.labels[d]}")
    for j in sorted(order):
        s, l = int(seg.start[lo + j]), int(seg.length[lo + j]); txt = tokenizer.decode(ds.flat[ds.offs[d] + s: ds.offs[d] + s + l].tolist())
        log(f"   top evidence @{s / ds.n_tok[d]:4.0%} score {seg.score[lo + j]:+7.3f}: {txt[:230]!r}")
    log(f"   anchor window ends with: ...{tokenizer.decode(wins[-1][-40:].tolist())!r}")

selection_report(ds_dev, "dev")
show_example(ds_dev, int(np.argmax(ds_dev.n_tok > N_SEL_WIN * CHUNK)))


In [ ]:
# ============================================================
# CELL 3 - MODEL-2 BACKBONE (verbatim architecture) + LoRA / LoRMA+ ADAPTERS + MIL HEAD
# ============================================================
@dataclass
class ModelConfig:
    vocab_size: int = 32000
    d_model: int = 640
    n_layers: int = 8
    n_heads: int = 10
    n_kv_heads: int = 2
    ffn_hidden: int = 2048
    dropout: float = 0.2
    max_len: int = 1088
    pad_id: int = 0
    unk_id: int = 1
    eos_id: int = 2

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        xf = x.float(); xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight).type_as(x)

def build_rope(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()

def apply_rope(x, cos, sin):
    x1, x2 = x.float().chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).type_as(x)

def repeat_kv(x, groups):
    if groups == 1: return x
    B, KV, T, hd = x.shape
    return x[:, :, None].expand(B, KV, groups, T, hd).reshape(B, KV * groups, T, hd)

class GQAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h, self.kv = cfg.n_heads, cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)
    def _qkv(self, x):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        return q, k, v
    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q, k, v = self._qkv(x)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        k, v = repeat_kv(k, self.groups), repeat_kv(v, self.groups)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up   = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)
    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.attn = RMSNorm(cfg.d_model), GQAttention(cfg)
        self.norm2, self.ffn = RMSNorm(cfg.d_model), SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x, cos, sin):
        x = x + self.drop(self.attn(self.norm1(x), cos, sin))
        return x + self.drop(self.ffn(self.norm2(x)))

class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        cos, sin = build_rope(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
    def _rope(self, T):
        return self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]

# ---------------- adapters ----------------
class AdaptedLinear(nn.Module):
    """frozen base Linear + low-rank trainable update.
       mode 'lora'       : y = W0 x + s * B A x                        (additive, Hu et al.)
       mode 'lorma_plus' : y = (I + s * B A) W0 x = y0 + s * B A y0    (multiplicative w/ additive inflation; my reading of LoRMA+, VERIFY vs the paper's repo)
       B is zero-initialised in both modes so training starts exactly at the pretrained model."""
    def __init__(self, base, r, alpha, p, mode):
        super().__init__()
        self.base, self.mode, self.scale = base, mode, alpha / r
        a_in = base.in_features if mode == "lora" else base.out_features
        self.A = nn.Parameter(torch.empty(r, a_in)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(p)
    def forward(self, x):
        y = self.base(x)
        z = x if self.mode == "lora" else y
        return y + F.linear(F.linear(self.drop(z), self.A), self.B) * self.scale

def apply_adapters(model):
    n = 0
    for blk in model.blocks:
        for parent in (blk.attn, blk.ffn):
            for name in LORA_TARGETS:
                lin = getattr(parent, name, None)
                if isinstance(lin, nn.Linear):
                    setattr(parent, name, AdaptedLinear(lin, LORA_R, LORA_ALPHA, LORA_DROPOUT, ADAPTER)); n += 1
    return n

def build_backbone():
    ck = torch.load(MODEL_CKPT, map_location="cpu", weights_only=False)
    cd = dict(ck.get("config", {})); cd["dropout"] = BACKBONE_DROPOUT
    cfg = ModelConfig(**{k: cd[k] for k in ModelConfig.__dataclass_fields__ if k in cd})
    assert cfg.vocab_size == VOCAB, f"tokenizer vocab {VOCAB} != model vocab {cfg.vocab_size}"
    assert CHUNK <= cfg.max_len, "CHUNK exceeds the RoPE table of the checkpoint"
    model = DecoderLM(cfg)
    model.load_state_dict(ck["model"], strict=True)          # strict: fails loudly if the architecture copy differs
    if "val_loss" in ck: log(f"loaded Model-2 weights (pretraining val loss {ck['val_loss']:.4f}, global step {ck.get('global_step')})")
    del ck
    for p in model.parameters(): p.requires_grad = False
    n_ad = apply_adapters(model)
    return model.to(DEVICE), cfg, n_ad

model, MCFG, n_adapted = build_backbone()
FEAT_DIM = 2 * MCFG.d_model
n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"Model-2: {n_total - n_train:,} frozen params | {n_adapted} adapted Linear layers ({ADAPTER}, r={LORA_R}) -> {n_train:,} trainable ({100*n_train/n_total:.2f}%)")

def backbone_features(model, ids, lens):
    """ids (N,T) right-padded, lens (N,). Causal attention => padding never influences real tokens, so no mask is needed.
    Per-window feature = [hidden state of the last real token ; mean of all real-token hidden states].
    (The last token is in-distribution for a next-token model. An appended EOS would NOT be: Model-2 never saw EOS in pretraining windows.)"""
    T = ids.size(1)
    cos, sin = model._rope(T)
    x = model.drop(model.embed(ids))
    for blk in model.blocks: x = blk(x, cos, sin)
    h = model.final_norm(x).float()
    ar = torch.arange(T, device=ids.device)[None]
    m = (ar < lens[:, None]).unsqueeze(-1).float()
    last = h[torch.arange(h.size(0), device=ids.device), lens - 1]
    mean = (h * m).sum(1) / lens[:, None].float()
    return torch.cat([last, mean], dim=-1)

class DocHead(nn.Module):
    """window features -> doc logits. Gated-attention multiple-instance pooling (Ilse et al. 2018) over windows,
    + learned position embeddings (first / relative position / last) + explicit tail-window branch + per-window auxiliary classifier."""
    def __init__(self, in_dim, d=256, att=128, p=0.15):
        super().__init__()
        self.norm, self.proj = nn.LayerNorm(in_dim), nn.Linear(in_dim, d)
        self.pos = nn.Embedding(NB + 2, d)
        self.drop = nn.Dropout(p)
        self.att_v, self.att_u, self.att_w = nn.Linear(d, att), nn.Linear(d, att), nn.Linear(att, 1)
        self.win_cls = nn.Linear(d, 2)
        self.doc_cls = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(p), nn.Linear(d, 2))
    def forward(self, feats, buckets, mask):
        h = F.gelu(self.proj(self.drop(self.norm(feats)))) + self.pos(buckets)
        h = self.drop(h)
        a = self.att_w(torch.tanh(self.att_v(h)) * torch.sigmoid(self.att_u(h))).squeeze(-1)
        a = a.masked_fill(~mask, -1e4)
        w = torch.softmax(a, dim=-1) * mask
        pooled = (w.unsqueeze(-1) * h).sum(1)
        last_idx = mask.long().sum(1) - 1
        h_last = h[torch.arange(h.size(0), device=h.device), last_idx]      # windows are ordered => last valid = tail window
        return self.doc_cls(torch.cat([pooled, h_last], dim=-1)), self.win_cls(h), w

head = DocHead(FEAT_DIM).to(DEVICE)
log(f"MIL head params: {sum(p.numel() for p in head.parameters()):,}")

def amp_sanity():
    """fp16 autocast vs fp32 on real windows (the Model-2 pretraining run had AMP switched off) - falls back to fp32 if they disagree"""
    global USE_AMP
    if not USE_AMP: return
    seqs = [ds_train.slice(i, 0, min(CHUNK, ds_train.n_tok[i])) for i in range(min(6, len(ds_train)))]
    T = max(len(s) for s in seqs); ids = np.zeros((len(seqs), T), dtype=np.int64); lens = np.array([len(s) for s in seqs])
    for i, s in enumerate(seqs): ids[i, :len(s)] = s
    ids, lens = torch.from_numpy(ids).to(DEVICE), torch.from_numpy(lens).to(DEVICE)
    model.eval()
    with torch.no_grad():
        f32 = backbone_features(model, ids, lens)
        with torch.autocast("cuda", dtype=torch.float16): f16 = backbone_features(model, ids, lens)
    cos_min = F.cosine_similarity(f32, f16.float(), dim=-1).min().item()
    ok = bool(torch.isfinite(f16).all()) and cos_min > 0.98
    log(f"AMP sanity: min cosine(fp32, fp16)={cos_min:.5f} finite={bool(torch.isfinite(f16).all())} -> {'fp16 autocast ON' if ok else 'DISABLED (fp32)'}")
    USE_AMP = ok
amp_sanity()


In [ ]:
# ============================================================
# CELL 4 - BATCHING, FEATURE EXTRACTION, DOC-LEVEL PREDICTION, METRICS
# ============================================================
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, accuracy_score, confusion_matrix

def pad_windows(seqs):
    T = max(len(s) for s in seqs)
    arr = np.zeros((len(seqs), T), dtype=np.int64)                   # pad id 0; irrelevant under causal attention
    for i, s in enumerate(seqs): arr[i, :len(s)] = s
    return arr, np.array([len(s) for s in seqs], dtype=np.int64)

def make_train_batch(ds, idxs, rng):
    seqs, doc_of, slot_of, rows, ys = [], [], [], [], []
    for b, di in enumerate(idxs):
        wins = select_context(ds, int(di), rng)                      # stochastic evidence selection each epoch = augmentation
        for slot, w in enumerate(wins):
            seqs.append(w); doc_of.append(b); slot_of.append(slot)
        rows.append(bucket_ids(np.arange(len(wins)), len(wins))); ys.append(int(ds.labels[di]))
    ids, lens = pad_windows(seqs)
    B, Km = len(idxs), max(len(r) for r in rows)
    bucket = np.zeros((B, Km), dtype=np.int64); mask = np.zeros((B, Km), dtype=bool)
    for i, r in enumerate(rows): bucket[i, :len(r)] = r; mask[i, :len(r)] = True
    t = lambda a: torch.from_numpy(a).to(DEVICE, non_blocking=True)
    return dict(ids=t(ids), lens=t(lens), doc_of=t(np.array(doc_of)), slot_of=t(np.array(slot_of)),
                bucket=t(bucket), mask=t(mask), y=t(np.array(ys, dtype=np.int64)))

@torch.no_grad()
def extract_features(ds, doc_indices, bs=None, tag="", picker=None):
    """run the (LoRA-adapted) backbone over the SELECTED-CONTEXT windows of every doc (deterministic selection);
       returns window features + bookkeeping (slot index, #windows) so any doc-level aggregator / ablation can be evaluated
       later without touching the GPU again. picker(ds, d) -> list of token arrays lets the ablations swap in other selectors."""
    bs = bs or INFER_BS
    picker = picker or select_context
    was_training = model.training
    model.eval(); head.eval()
    seqs, ptr, gidx, nall = [], [0], [], []
    for di in doc_indices:
        wins = picker(ds, int(di))
        for g, w in enumerate(wins): seqs.append(w); gidx.append(g)
        ptr.append(len(seqs)); nall.append(len(wins))
    feats = np.zeros((len(seqs), FEAT_DIM), dtype=np.float32)
    t0 = time.time(); nb = (len(seqs) + bs - 1) // bs
    for bi in range(nb):
        ids, lens = pad_windows(seqs[bi * bs:(bi + 1) * bs])
        with amp_ctx():
            f = backbone_features(model, torch.from_numpy(ids).to(DEVICE), torch.from_numpy(lens).to(DEVICE))
        feats[bi * bs: bi * bs + len(lens)] = f.float().cpu().numpy()
        if tag and (bi % 100 == 0 or bi == nb - 1):
            log(f"    [{tag}] batch {bi+1}/{nb} | {len(seqs)/max(time.time()-t0,1e-6):.0f} windows/s (avg)" if bi else f"    [{tag}] {len(seqs):,} windows from {len(doc_indices):,} docs")
    if was_training: model.train(); head.train()
    return feats, np.array(ptr), np.array(gidx), np.array(nall)

@torch.no_grad()
def doc_probs(feats, ptr, gidx, nall, keep=None, agg="att", collect_att=False):
    """P(class=1) per document from cached window features.
       keep(g, n_all)->bool mask restricts WHICH windows the head may see (section ablations); agg='winmean' = mean of window logits (no attention)"""
    head.eval()
    ND = len(ptr) - 1; probs = np.zeros(ND, dtype=np.float64); att = []
    for b0 in range(0, ND, DOC_BS):
        docs = list(range(b0, min(ND, b0 + DOC_BS))); rows = []
        for d in docs:
            sel = np.arange(ptr[d], ptr[d + 1])
            if keep is not None:
                m = np.asarray(keep(gidx[sel], int(nall[d])), dtype=bool)
                sel = sel[m] if m.any() else sel[-1:]
            rows.append(sel)
        Km = max(len(r) for r in rows)
        Fb = np.zeros((len(docs), Km, FEAT_DIM), dtype=np.float32); Bk = np.zeros((len(docs), Km), dtype=np.int64); Mk = np.zeros((len(docs), Km), dtype=bool)
        for i, (d, r) in enumerate(zip(docs, rows)):
            Fb[i, :len(r)] = feats[r]; Bk[i, :len(r)] = bucket_ids(gidx[r], int(nall[d])); Mk[i, :len(r)] = True
        Ft, Bt, Mt = (torch.from_numpy(a).to(DEVICE) for a in (Fb, Bk, Mk))
        logits, wlog, w = head(Ft, Bt, Mt)
        if agg == "winmean":
            logits = (wlog * Mt.unsqueeze(-1)).sum(1) / Mt.sum(1, keepdim=True)
        probs[b0:b0 + len(docs)] = torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()
        if collect_att:
            wn = w.cpu().numpy()
            for i, r in enumerate(rows): att.append((Bk[i, :len(r)], wn[i, :len(r)]))
    return (probs, att) if collect_att else probs

# ---------------- metrics ----------------
def macro_f1_np(y, pred):
    f = []
    for c in (0, 1):
        tp = np.sum((pred == c) & (y == c)); fp = np.sum((pred == c) & (y != c)); fn = np.sum((pred != c) & (y == c))
        p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
        f.append(2 * p * r / (p + r) if p + r else 0.0)
    return float(np.mean(f))

def tune_threshold(y, p, lo=0.20, hi=0.80, step=0.01, smooth=5):
    """threshold maximising Macro-F1; the F1 curve is smoothed so one lucky threshold on a small dev set is not chosen"""
    grid = np.arange(lo, hi + 1e-9, step)
    f = np.array([macro_f1_np(y, (p >= t).astype(int)) for t in grid])
    fs = np.convolve(np.pad(f, smooth // 2, mode="edge"), np.ones(smooth) / smooth, mode="valid")
    i = int(np.argmax(fs)); return float(grid[i]), float(f[i]), float(macro_f1_np(y, (p >= 0.5).astype(int)))

def full_metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    P, R, Fm, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    pc, rc, fc, _ = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)
    try: auc = roc_auc_score(y, p)
    except ValueError: auc = float("nan")
    return dict(thr=thr, macro_f1=Fm, macro_precision=P, macro_recall=R, accuracy=accuracy_score(y, pred), auc=auc,
                p0=pc[0], r0=rc[0], f1_0=fc[0], p1=pc[1], r1=rc[1], f1_1=fc[1], confusion=confusion_matrix(y, pred, labels=[0, 1]).tolist())

# fixed quick-validation subset of the (leak-free) dev set
_rs = np.random.default_rng(SEED + 1)
QUICK_IDX = np.sort(_rs.choice(len(ds_dev), min(N_QUICK, len(ds_dev)), replace=False))

def quick_eval():
    was_training = model.training
    f, ptr, g, na = extract_features(ds_dev, QUICK_IDX)
    p = doc_probs(f, ptr, g, na); y = ds_dev.labels[QUICK_IDX]
    thr, f1_tuned, f1_05 = tune_threshold(y, p)
    try: auc = roc_auc_score(y, p)
    except ValueError: auc = float("nan")
    if was_training: model.train(); head.train()
    return dict(auc=auc, f1_05=f1_05, f1_tuned=f1_tuned, thr=thr)


In [ ]:
# ============================================================
# CELL 5 - TRAINING (time-boxed, resumable, class-weighted, LoRA + MIL head)
# ============================================================
trainable = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW([{"params": trainable, "lr": LR_LORA, "weight_decay": 0.0},
                         {"params": list(head.parameters()), "lr": LR_HEAD, "weight_decay": WD_HEAD}], betas=(0.9, 0.999))
BASE_LRS = [LR_LORA, LR_HEAD]
try:    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
except Exception: scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

import shutil
S = dict(epoch=0, doc_ptr=0, step=0, train_sec=0.0, docs=0, wins=0, best_auc=-1.0, best_step=-1)
LATEST, BEST = CKPT / "latest_cjpe.pt", CKPT / "best_adapter.pt"
RUN_CFG = dict(adapter=ADAPTER, r=LORA_R, alpha=LORA_ALPHA, chunk=CHUNK, context="selective", n_sel_win=N_SEL_WIN, head_tok=HEAD_TOK, tail_tok=TAIL_TOK, seg=[SEG_MIN, SEG_MAX], lowercase=LOWERCASE, feat_dim=FEAT_DIM, targets=list(LORA_TARGETS))

def adapter_state():
    return {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}

def load_adapter_state(lora_sd, head_sd):
    own = dict(model.named_parameters()); miss = 0
    for n, v in lora_sd.items():
        if n in own: own[n].data.copy_(v.to(own[n].device))
        else: miss += 1
    head.load_state_dict(head_sd)
    if miss: log(f"warning: {miss} adapter tensors in the checkpoint did not match this model")

def save_latest():
    atomic_save({"lora": adapter_state(), "head": head.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                 "state": dict(S), "cfg": RUN_CFG}, LATEST)

def save_best(res):
    atomic_save({"lora": adapter_state(), "head": head.state_dict(), "cfg": RUN_CFG, "quick": res, "step": S["step"]}, BEST)

def try_resume():
    if SMOKE: return False
    cands = [p for p in [str(LATEST)] + find_files("latest_cjpe.pt", [INPUT_ROOT]) if os.path.exists(p)]
    best = None
    for p in cands:
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False)
            if ck.get("cfg", {}).get("adapter") == ADAPTER and ck["cfg"].get("feat_dim") == FEAT_DIM and ck["cfg"].get("context") == "selective" and ck["cfg"].get("n_sel_win") == N_SEL_WIN:
                if best is None or ck["state"]["train_sec"] > best[1]["state"]["train_sec"]: best = (p, ck)
        except Exception as e:
            log(f"could not read {p}: {e}")
    if best is None: return False
    p, ck = best
    load_adapter_state(ck["lora"], ck["head"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"]); S.update(ck["state"])
    log(f"RESUMED from {p}: step {S['step']}, epoch {S['epoch']}, {S['train_sec']/60:.0f} min already trained, best quick AUC {S['best_auc']:.4f}")
    # carry the previous best adapter over (it may live in a read-only /kaggle/input dir)
    prev_best = os.path.join(os.path.dirname(p), "best_adapter.pt")
    if not BEST.exists() and os.path.exists(prev_best): shutil.copy(prev_best, BEST)
    return True

def lr_mult(step, progress):
    warm = min(1.0, (step + 1) / WARMUP_STEPS)
    return warm * (MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * min(progress, 1.0))))

def micro_step(idxs, rng):
    b = make_train_batch(ds_train, idxs, rng)
    with amp_ctx():
        feats = backbone_features(model, b["ids"], b["lens"])
    feats = feats.float()
    B, Km = b["mask"].shape
    padded = torch.zeros(B, Km, feats.size(-1), device=feats.device).index_put((b["doc_of"], b["slot_of"]), feats)
    logits, wlog, _ = head(padded, b["bucket"], b["mask"])
    loss_doc = F.cross_entropy(logits, b["y"], weight=CLASS_W)                                   # class-imbalance: inverse-frequency weights
    wy = b["y"][:, None].expand(B, Km)[b["mask"]]
    loss_aux = F.cross_entropy(wlog[b["mask"]], wy, weight=CLASS_W, label_smoothing=0.1)         # every window sees the doc label (noisy -> smoothed)
    loss = loss_doc + AUX_W * loss_aux
    if not torch.isfinite(loss): return float("nan"), 0
    scaler.scale(loss / ACCUM).backward()
    return float(loss.item()), int(b["ids"].size(0))

def train():
    N = len(ds_train); budget = TRAIN_BUDGET_MIN * 60.0
    session_cap = max(0.0, (SESSION_LIMIT_H * 60 - RESERVE_MIN) * 60 - (time.time() - T0))
    t_sess, sec0 = time.time(), S["train_sec"]
    last_ckpt = last_eval = t_log = time.time(); w_log, losses, n_oom, stop, docs_sess = 0, [], 0, None, 0
    model.train(); head.train()
    log(f"training: budget {TRAIN_BUDGET_MIN} min total ({sec0/60:.0f} done) | this session may use {session_cap/60:.0f} min | "
        f"{MICRO_DOCS*ACCUM} docs/step x <={N_SEL_WIN} selected-context windows | {N:,} train docs")
    while stop is None and S["epoch"] < MAX_EPOCHS:
        perm = np.random.default_rng(SEED + S["epoch"]).permutation(N)
        rng = np.random.default_rng(SEED * 7 + S["epoch"] * 1000 + S["step"])
        while S["doc_ptr"] < N:
            sess = time.time() - t_sess; elapsed = sec0 + sess; S["train_sec"] = elapsed
            progress = max(elapsed / budget, (S["epoch"] * N + S["doc_ptr"]) / (MAX_EPOCHS * N))
            if progress >= 1.0: stop = "budget reached"; break
            if sess >= session_cap: stop = "session limit reached - resume next session"; break
            m = lr_mult(S["step"], progress)
            for g, base in zip(opt.param_groups, BASE_LRS): g["lr"] = base * m
            opt.zero_grad(set_to_none=True); nmb = 0
            for _ in range(ACCUM):
                if S["doc_ptr"] >= N: break
                idxs = perm[S["doc_ptr"]: S["doc_ptr"] + MICRO_DOCS]; S["doc_ptr"] += len(idxs)
                try:
                    l, nw = micro_step(idxs, rng)
                except OOM_ERR:
                    n_oom += 1; opt.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                    log(f"  CUDA OOM #{n_oom} - skipped a micro-batch (lower MICRO_DOCS or N_SEL_WIN if this repeats)")
                    if n_oom > 5: raise
                    continue
                if nw: losses.append(l); w_log += nw; S["docs"] += len(idxs); S["wins"] += nw; docs_sess += len(idxs); nmb += 1
            if nmb == 0: continue
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable + list(head.parameters()), GRAD_CLIP)
            scaler.step(opt); scaler.update(); S["step"] += 1
            if S["step"] % LOG_EVERY == 0:
                dt = time.time() - t_log; wps = w_log / max(dt, 1e-6)
                eta_epoch = (N - S["doc_ptr"]) / max(docs_sess / max(sess, 1e-6), 1e-6)
                log(f"ep {S['epoch']} step {S['step']:5d} | docs {S['doc_ptr']:6,}/{N:,} | loss {np.mean(losses[-LOG_EVERY*ACCUM:]):.4f} | "
                    f"lr x{m:.2f} | {wps:5.1f} win/s | trained {fmt_t(elapsed)}/{fmt_t(budget)} | epoch ETA {fmt_t(eta_epoch)}")
                t_log, w_log = time.time(), 0
            now = time.time()
            if now - last_eval >= EVAL_EVERY_MIN * 60:
                r = quick_eval(); last_eval = time.time()
                better = (not np.isnan(r["auc"])) and r["auc"] > S["best_auc"]
                if better: S["best_auc"], S["best_step"] = r["auc"], S["step"]; save_best(r)
                log(f"  >> quick-dev ({len(QUICK_IDX)} docs, selective context): AUC {r['auc']:.4f} | macroF1@0.5 {r['f1_05']:.4f} | "
                    f"macroF1@thr{r['thr']:.2f} {r['f1_tuned']:.4f} {'(NEW BEST)' if better else ''}")
                t_log = time.time()
            if now - last_ckpt >= CKPT_EVERY_MIN * 60: save_latest(); last_ckpt = now
        if stop is None: S["epoch"] += 1; S["doc_ptr"] = 0
    # final bookkeeping
    S["train_sec"] = sec0 + (time.time() - t_sess)
    r = quick_eval()
    better = (not np.isnan(r["auc"])) and r["auc"] > S["best_auc"]
    if better or not BEST.exists(): S["best_auc"], S["best_step"] = r["auc"], S["step"]; save_best(r)
    log(f"  >> final quick-dev: AUC {r['auc']:.4f} | macroF1@0.5 {r['f1_05']:.4f} | macroF1@thr{r['thr']:.2f} {r['f1_tuned']:.4f} {'(NEW BEST)' if better else ''}")
    save_latest()
    log(f"training stopped: {stop or 'epochs finished'} | steps {S['step']} | docs seen {S['docs']:,} | windows {S['wins']:,} | best quick AUC {S['best_auc']:.4f} @ step {S['best_step']}")

if MODE == "train_predict":
    try_resume()
    train()
else:
    log("MODE=predict_only: skipping training")


In [ ]:
# ============================================================
# CELL 6 - FULL DEV EVALUATION: threshold tuning, section ablations, attention analysis, error analysis
# (dev = dev.json minus every test-case ID, so nothing here touches test labels)
# ============================================================
# load the best adapter (by quick-dev AUC); fall back to whatever is currently in memory
_best = None
for p in [str(BEST)] + find_files("best_adapter.pt", [INPUT_ROOT]):
    if os.path.exists(p):
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False)
            if ck["cfg"].get("adapter") == ADAPTER and ck["cfg"].get("feat_dim") == FEAT_DIM and ck["cfg"].get("context") == "selective" and ck["cfg"].get("n_sel_win") == N_SEL_WIN: _best = (p, ck); break
        except Exception as e:
            log(f"could not read {p}: {e}")
if _best:
    load_adapter_state(_best[1]["lora"], _best[1]["head"])
    log(f"evaluating best adapter from {_best[0]} (step {_best[1].get('step')}, quick AUC {_best[1]['quick']['auc']:.4f})")
else:
    assert MODE == "train_predict", "predict_only needs a best_adapter.pt (attach the training run's output as an input)"
    log("no best_adapter.pt found - using the weights currently in memory")

t0 = time.time()
dev_idx = np.arange(len(ds_dev))
f_dev, ptr_dev, g_dev, n_dev = extract_features(ds_dev, dev_idx, tag="dev")
log(f"dev feature extraction: {(time.time()-t0)/60:.1f} min")
np.savez(OUT / "dev_window_feats.npz", feats=f_dev, ptr=ptr_dev, gidx=g_dev, nall=n_dev, y=ds_dev.labels)   # reusable by later aggregator experiments
y_dev = ds_dev.labels
p_dev = doc_probs(f_dev, ptr_dev, g_dev, n_dev)

THR, f1_tuned, f1_05 = tune_threshold(y_dev, p_dev)
if abs(THR - 0.5) > 0.15: log(f"warning: tuned threshold {THR:.2f} is far from 0.5 - check for train/dev shift before trusting it")
M05, MT = full_metrics(y_dev, p_dev, 0.5), full_metrics(y_dev, p_dev, THR)
log("\n" + "=" * 90 + f"\nDEV RESULTS ({len(y_dev):,} docs, class balance {np.bincount(y_dev, minlength=2).tolist()})\n" + "=" * 90)
for name, m in (("threshold 0.50", M05), (f"tuned thr {THR:.2f}", MT)):
    log(f"{name:15s} | Macro-F1 {m['macro_f1']:.4f} | Macro-P {m['macro_precision']:.4f} | Macro-R {m['macro_recall']:.4f} | Acc {m['accuracy']:.4f} | AUC {m['auc']:.4f}")
    log(f"{'':15s} | class0 P/R/F1 {m['p0']:.3f}/{m['r0']:.3f}/{m['f1_0']:.3f} | class1 P/R/F1 {m['p1']:.3f}/{m['r1']:.3f}/{m['f1_1']:.3f} | confusion [[TN,FP],[FN,TP]] = {m['confusion']}")

# ---------- ablations (inference-time: the trained head sees only the selected windows) ----------
ABL = [("all selected windows (attention)",                       None,                      "att"),
       ("all selected windows (mean of window logits)",           None,                      "winmean"),
       ("anchor window only (head + tail + top evidence)",        lambda g, n: g == n - 1,   "att"),
       ("evidence window(s) only (no tail guarantee)",            lambda g, n: g < n - 1,    "att")]
rows = []
for name, keep, agg in ABL:
    p = p_dev if (keep is None and agg == "att") else doc_probs(f_dev, ptr_dev, g_dev, n_dev, keep=keep, agg=agg)
    thr_, f_t, f_05 = tune_threshold(y_dev, p)
    rows.append(dict(setting=name, AUC=roc_auc_score(y_dev, p), MacroF1_at_0_5=f_05, MacroF1_tuned_thr=f_t, thr=thr_))
abl_df = pd.DataFrame(rows).round(4)
log("\nWINDOW ABLATION (same trained model, restricted input; 'tuned thr' column is optimistic - tuned on dev itself)")
log(abl_df.to_string(index=False))
abl_df.to_csv(OUT / "ablation_sections.csv", index=False)

# ---------- does the evidence selector matter? same trained model + head, other ways to spend the SAME token budget ----------
def _tail_only(ds, d):
    n, cap, a0 = int(ds.n_tok[d]), N_SEL_WIN * CHUNK, int(ds.offs[d]); lo = max(0, n - cap)
    return [ds.flat[a0 + lo + int(s): a0 + lo + int(s) + min(CHUNK, (n - lo) - int(s))] for s in all_starts(n - lo, CHUNK, CHUNK)]
_rng_ab = np.random.default_rng(SEED)
SELECTORS = [("evidence selector (this notebook)", None),
             ("random segments + same head/tail",  lambda ds, d: select_context(ds, d, _rng_ab, mode="random")),
             ("last-N tokens (plain truncation)",  _tail_only)]
rows = []
for name, pk in SELECTORS:
    p = p_dev if pk is None else doc_probs(*extract_features(ds_dev, dev_idx, picker=pk))
    thr_, f_t, f_05 = tune_threshold(y_dev, p)
    rows.append(dict(selector=name, AUC=roc_auc_score(y_dev, p), MacroF1_at_0_5=f_05, MacroF1_tuned_thr=f_t, thr=thr_))
sel_df = pd.DataFrame(rows).round(4)
log("\nSELECTOR ABLATION (identical token budget; selector swapped at inference only)"); log(sel_df.to_string(index=False))
sel_df.to_csv(OUT / "ablation_selector.csv", index=False)

# ---------- where does the attention go? ----------
_, att = doc_probs(f_dev, ptr_dev, g_dev, n_dev, collect_att=True)
mass = np.zeros(NB + 2); cnt_ = np.zeros(NB + 2)
for b, w in att:
    for bi, wi in zip(b, w): mass[bi] += wi; cnt_[bi] += 1
labels_b = ["first slot / only window"] + [f"middle slot bucket {i+1}" for i in range(NB)] + ["anchor window (head+tail+evidence)"]
att_df = pd.DataFrame({"window position": labels_b, "mean attention weight": mass / np.maximum(cnt_, 1), "share of total attention": mass / mass.sum(), "n windows": cnt_.astype(int)}).round(4)
log("\nATTENTION BY WINDOW SLOT (dev)"); log(att_df.to_string(index=False)); att_df.to_csv(OUT / "attention_by_position.csv", index=False)

# ---------- error analysis ----------
pred_dev = (p_dev >= THR).astype(int); ntok = ds_dev.n_tok
grp = np.where((pred_dev == 1) & (y_dev == 1), "TP", np.where((pred_dev == 0) & (y_dev == 0), "TN", np.where((pred_dev == 1) & (y_dev == 0), "FP", "FN")))
err_df = pd.DataFrame({"group": grp, "tokens": ntok, "windows": n_dev, "p1": p_dev, "mixed_case_src": ds_dev.has_upper})
summ = err_df.groupby("group").agg(n=("tokens", "size"), mean_tokens=("tokens", "mean"), median_tokens=("tokens", "median"), mean_windows=("windows", "mean"), mean_p1=("p1", "mean"), mixed_case_share=("mixed_case_src", "mean")).round(2)
log("\nERROR ANALYSIS at tuned threshold (FP = predicted 1, true 0 | FN = predicted 0, true 1)"); log(summ.to_string())
q = pd.qcut(ntok, 4, duplicates="drop"); er = pd.DataFrame({"len_quartile": q, "wrong": pred_dev != y_dev, "y": y_dev}).groupby("len_quartile", observed=True).agg(docs=("wrong", "size"), error_rate=("wrong", "mean"), class1_share=("y", "mean")).round(3)
log("\nERROR RATE BY DOCUMENT LENGTH QUARTILE (tokens)"); log(er.to_string())
summ.to_csv(OUT / "error_groups.csv"); er.to_csv(OUT / "error_by_length.csv")

pd.DataFrame({"id": ds_dev.ids, "label": y_dev, "p1": p_dev, "pred": pred_dev, "tokens": ntok, "windows": n_dev}).to_csv(OUT / "dev_probs.csv", index=False)
json.dump({"threshold": THR, "dev_at_0.5": M05, "dev_at_tuned": MT, "run": RUN_CFG, "train_state": S if "S" in globals() else None}, open(OUT / "dev_metrics.json", "w"), indent=1, default=float)


In [ ]:
# ============================================================
# CELL 7 - TEST PREDICTION + SUBMISSION FILE
# ============================================================
t0 = time.time()
test_idx = np.arange(len(ds_test))
f_te, ptr_te, g_te, n_te = extract_features(ds_test, test_idx, tag="test")
log(f"test feature extraction: {(time.time()-t0)/60:.1f} min")
np.savez(OUT / "test_window_feats.npz", feats=f_te, ptr=ptr_te, gidx=g_te, nall=n_te)
p_te = doc_probs(f_te, ptr_te, g_te, n_te)
pred_te = (p_te >= THR).astype(int)

pd.DataFrame({"id": ds_test.ids, "p1": p_te, "pred": pred_te}).to_csv(OUT / "test_probs.csv", index=False)   # probabilities kept for later ensembling
submission = [{"id": i, "pred": int(v)} for i, v in zip(ds_test.ids, pred_te)]

# ---- validation of the required format ----
assert len(submission) == len(ds_test) == len(set(ds_test.ids)), "one prediction per test case required"
assert all(set(s) == {"id", "pred"} and type(s["pred"]) is int and s["pred"] in (0, 1) for s in submission)
if not SMOKE:
    ref = [str(r["id"]) for r in _records("test")]
    assert [s["id"] for s in submission] == ref, "submission ids/order differ from test.json"
sub_path = SUB / "prediction.json"
with open(sub_path, "w", encoding="utf-8") as f: json.dump(submission, f)
import zipfile
with zipfile.ZipFile(SUB / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z: z.write(sub_path, "prediction.json")

log("\n" + "=" * 90)
log(f"SUBMISSION WRITTEN: {sub_path}   ({len(submission):,} predictions{' - SMOKE RUN, NOT A VALID SUBMISSION' if SMOKE else ''})")
log(f"predicted class balance: class0={int((pred_te==0).sum()):,} ({100*(pred_te==0).mean():.1f}%) | class1={int((pred_te==1).sum()):,} ({100*(pred_te==1).mean():.1f}%) | threshold {THR:.2f}")
log(f"dev Macro-F1 (leak-free dev, tuned thr): {MT['macro_f1']:.4f} | at 0.5: {M05['macro_f1']:.4f} | AUC {MT['auc']:.4f}")
log(f"total wall time: {(time.time()-T0)/60:.0f} min | outputs in {WORK}")
log("=" * 90)
